# Housing Price Prediction - Amsterdam
This notebook builds ML models to predict housing prices in Amsterdam using the Ground Control dataset.

In [ ]:
# Importsimport pandas as pdimport numpy as npimport sqlite3import matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler, OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipelinefrom sklearn.linear_model import LinearRegression, Ridgefrom sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressorfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scoreimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load data
DB_PATH = 'ground_control.db'
conn = sqlite3.connect(DB_PATH)
listings = pd.read_sql_query('SELECT * FROM listings', conn)
neighbourhood_stats = pd.read_sql_query('SELECT * FROM neighbourhood_stats', conn)
print(f'Loaded {len(listings)} listings')

In [ ]:
# Basic infoprint(f'Shape: {listings.shape}')print(f'\nColumns: {listings.columns.tolist()}')

## Exploratory Data Analysis

In [ ]:
# Price distributionfig, ax = plt.subplots(1, 1, figsize=(10, 5))listings['price_numeric'].hist(bins=50, ax=ax, edgecolor='black')ax.set_xlabel('Price (€)')ax.set_ylabel('Frequency')ax.set_title('Distribution of Listing Prices')plt.tight_layout()plt.show()print(f'\nPrice Statistics:\n{listings["price_numeric"].describe()}')

In [ ]:
# Living area vs Priceplt.figure(figsize=(10, 6))plt.scatter(listings['living_area'], listings['price_numeric'], alpha=0.3)plt.xlabel('Living Area (m²)')plt.ylabel('Price (€)')plt.title('Living Area vs Price')plt.tight_layout()plt.show()print(f'Correlation: {listings["living_area"].corr(listings["price_numeric"]):.3f}')

## Feature Engineering

In [ ]:
# Drop irrelevant columnscols_to_drop = ['address', 'city', 'listing_url', 'detail_url', 'agent_name', 'agent_url', 'image_url', 'plot_area', 'previous_price', 'labels', 'listing_type', 'is_active', 'availability_status', 'price', 'global_id']df = listings.drop(columns=[c for c in cols_to_drop if c in listings.columns])df = df.dropna(subset=['price_numeric'])print(f'After cleaning: {len(df)} rows')

In [ ]:
# Feature: price_per_m2 (for analysis)df['price_per_m2'] = df['price_numeric'] / df['living_area']print(f'Price per m2 - Mean: €{df["price_per_m2"].mean():,.0f}, Median: €{df["price_per_m2"].median():,.0f}')

In [ ]:
# Feature: days_on_marketdf['first_seen'] = pd.to_datetime(df['first_seen'])df['last_seen'] = pd.to_datetime(df['last_seen'])df['days_on_market'] = (df['last_seen'] - df['first_seen']).dt.daysprint(f'Days on market - Mean: {df["days_on_market"].mean():.1f}')

In [ ]:
# Feature: postcode_prefixdf['postcode_prefix'] = df['postcode'].str[:4]print(f'Unique postcode prefixes: {df["postcode_prefix"].nunique()}')

In [ ]:
# Feature: energy_label ordinalenergy_map = {'A++++': 12, 'A+++': 11, 'A++': 10, 'A+': 9, 'A': 8, 'B': 7, 'C': 6, 'D': 5, 'E': 4, 'F': 3, 'G': 2, 'unknown': 0}df['energy_score'] = df['energy_label'].map(energy_map).fillna(0)print('Energy score distribution:\n', df['energy_score'].value_counts().sort_index())

In [ ]:
# Feature: neighbourhood_stats mergedf = df.merge(neighbourhood_stats[['neighbourhood', 'avg_price_m2', 'median_price']], on='neighbourhood', how='left')city_avg = neighbourhood_stats['avg_price_m2'].mean()df['avg_price_m2'] = df['avg_price_m2'].fillna(city_avg)df['median_price'] = df['median_price'].fillna(df['median_price'].median())print(f'After merge: {len(df)} rows')

## Train/Test Split

In [ ]:
# Define features and targetnumeric_features = ['living_area', 'bedrooms', 'days_on_market', 'energy_score', 'avg_price_m2', 'median_price']categorical_features = ['construction_type', 'postcode_prefix', 'is_project']X = df[numeric_features + categorical_features].copy()y = df['price_numeric']X[numeric_features] = X[numeric_features].fillna(X[numeric_features].median())X[categorical_features] = X[categorical_features].fillna('unknown')print(f'Features: {X.shape[1]}, Target: {y.shape[0]}')

In [ ]:
# Train/test splitX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Model Training

In [ ]:
# Preprocessing pipelinepreprocessor = ColumnTransformer(    transformers=[        ('num', StandardScaler(), numeric_features),        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)    ])X_train_transformed = preprocessor.fit_transform(X_train)X_test_transformed = preprocessor.transform(X_test)print(f'Transformed shape: {X_train_transformed.shape}')

In [ ]:
# Evaluation helperdef evaluate_model(model, name):    model.fit(X_train_transformed, y_train)    y_pred_train = model.predict(X_train_transformed)    y_pred_test = model.predict(X_test_transformed)    train_mae = mean_absolute_error(y_train, y_pred_train)    test_mae = mean_absolute_error(y_test, y_pred_test)    train_r2 = r2_score(y_train, y_pred_train)    test_r2 = r2_score(y_test, y_pred_test)    print(f'\n{name}:')    print(f'  Train MAE: €{train_mae:,.0f}')    print(f'  Test MAE:  €{test_mae:,.0f}')    print(f'  Train R²:  {train_r2:.3f}')    print(f'  Test R²:   {test_r2:.3f}')    return model, {'name': name, 'test_mae': test_mae, 'test_r2': test_r2}

In [ ]:
# Linear Regressionlr_model, lr_results = evaluate_model(LinearRegression(), 'Linear Regression')

In [ ]:
# Ridge Regressionridge_model, ridge_results = evaluate_model(Ridge(alpha=1.0), 'Ridge Regression')

In [ ]:
# Random Forestrf_model, rf_results = evaluate_model(RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1), 'Random Forest')

In [ ]:
# Gradient Boostinggb_model, gb_results = evaluate_model(GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42), 'Gradient Boosting')

In [ ]:
# Model comparisonresults_df = pd.DataFrame([lr_results, ridge_results, rf_results, gb_results])results_df = results_df.sort_values('test_r2', ascending=False)print('\nModel Comparison (sorted by Test R²):')print(results_df.to_string(index=False))

## Model Analysis

In [ ]:
# Feature importance (Random Forest)feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))importances = rf_model.feature_importances_feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)plt.figure(figsize=(10, 6))plt.barh(feat_imp['feature'], feat_imp['importance'])plt.xlabel('Importance')plt.title('Top 15 Feature Importances')plt.tight_layout()plt.show()

In [ ]:
# Predicted vs Actualy_pred = gb_model.predict(X_test_transformed)plt.figure(figsize=(10, 6))plt.scatter(y_test, y_pred, alpha=0.3)plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)plt.xlabel('Actual Price (€)')plt.ylabel('Predicted Price (€)')plt.title('Predicted vs Actual Prices')plt.tight_layout()plt.show()

## Save Model

In [ ]:
# Save full pipelineimport joblibfull_pipeline = Pipeline([    ('preprocessor', preprocessor),    ('regressor', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42))])full_pipeline.fit(X_train, y_train)joblib.dump(full_pipeline, 'housing_price_model.pkl')print('Model saved to housing_price_model.pkl')

## How to UseLoad the model and predict:```pythonimport joblibmodel = joblib.load('housing_price_model.pkl')predictions = model.predict(new_listing_features)```## Intrinsic ValueCompare predicted price (model's estimate of fair value) vs listing price to find undervalued properties!